# RustPower Lesson 3: Transactions, Batch Edits, and Rollback 🧱

In large-scale studies you often add or modify hundreds of elements at once.
Calling Python→Rust once per element is slow, and a half-applied modification
is worse than a slow one.

`grid.edit()` solves both: it is a **transaction**. Commands are buffered in the
Harvard command queue and fused-applied once on commit (each entity's components
land in a single insert — no archetype fragmentation), and an exception inside
the block rolls everything back.

In [1]:
import rustpower
import time
import numpy as np

grid = rustpower.PowerGrid()
grid.set_base(sn_mva=100.0)

## 1. Batch construction in one transaction

A 1000-bus chain network, built in a single `edit()` block. The commit marks the
topology dirty; the next `solve()` rebuilds automatically.

In [2]:
print("Batch adding 1000 buses and lines...")
start = time.time()

with grid.edit() as e:
    prev_id, _ = e.add_bus(vn_kv=110.0)
    e.add_ext_grid(bus=prev_id, vm_pu=1.0)

    for i in range(1, 1000):
        new_id, _ = e.add_bus(vn_kv=110.0)
        e.add_line(from_bus=prev_id, to_bus=new_id, length_km=1.0,
                   r_ohm_per_km=0.01, x_ohm_per_km=0.03)
        prev_id = new_id

    # a feeder load at the far end of the chain
    e.add_load(bus=prev_id, p_mw=2.0, q_mvar=0.5)

build_ms = (time.time() - start) * 1000
print(f"Transaction committed in {build_ms:.2f} ms")
print(f"Final grid: {grid.n_bus} buses, {grid.n_line} lines.")

report = grid.solve()
print(report)

Batch adding 1000 buses and lines...
Transaction committed in 2.35 ms
Final grid: 1000 buses, 999 lines.
SolveReport(converged=true, iterations=1, runtime_ms=3.865, rebuild='full')


## 2. Rollback: exceptions abort the transaction

If anything goes wrong mid-edit, the world is left exactly as it was at the
transaction start. No half-built topology, ever.

In [3]:
n_before = grid.n_bus

try:
    with grid.edit() as e:
        e.add_bus(vn_kv=110.0)
        e.add_bus(vn_kv=110.0)
        raise ValueError("validation failed half-way through!")
except ValueError as ex:
    print(f"Aborted: {ex}")

print(f"Bus count: {n_before} before, {grid.n_bus} after (unchanged)")

report = grid.solve()
print(f"Grid still solves: {bool(report)} (rebuild={report.rebuild!r})")

Aborted: validation failed half-way through!
Bus count: 1000 before, 1000 after (unchanged)
Grid still solves: True (rebuild='incremental')


## 3. Case data vs results

RustPower distinguishes the **case data** (your inputs) from the **results**
(the solver output):

* `display_case_buses()` — inputs: vn_kv, node types, names.
* `grid.res_bus` — solution: vm_pu, va_degree, p_mw, q_mvar.

In [4]:
print("--- Inputs ---")
print(grid.display_case_buses().head())

print("\n--- Solution ---")
print(grid.res_bus.head())

--- Inputs ---


   bus_id   name type  vn_kv  vm_min_pu  vm_max_pu
0       1  bus_1   PQ  110.0        0.9        1.1
1       2  bus_2   PQ  110.0        0.9        1.1
2       3  bus_3   PQ  110.0        0.9        1.1
3       4  bus_4   PQ  110.0        0.9        1.1
4       5  bus_5   PQ  110.0        0.9        1.1

--- Solution ---
   bus_id                v_pu     vm_pu  va_degree          p_mw        q_mvar
0       0  1.000000+0.000000j  1.000000   0.000000 -2.003529e+00 -5.105879e-01
1       1  0.999997-0.000005j  0.999997  -0.000260  1.136857e-10 -1.818989e-10
2       2  0.999994-0.000009j  0.999994  -0.000521  1.591623e-10  1.818964e-10
3       3  0.999991-0.000014j  0.999991  -0.000781 -1.818973e-10  2.480440e-15
4       4  0.999988-0.000018j  0.999988  -0.001042  6.820883e-11 -1.364239e-10


## 4. High-speed voltage access

For machine learning or optimization loops, `grid.v` returns the raw complex
voltage vector as a NumPy array (original bus order) without DataFrame overhead.

In [5]:
v_raw = grid.v
print(f"Raw voltage vector type: {type(v_raw)}, shape {v_raw.shape}")
print(f"First 5 voltages (complex p.u.):\n{v_raw[:5]}")
print(f"Magnitudes:\n{np.abs(v_raw[:5])}")

Raw voltage vector type: <class 'numpy.ndarray'>, shape (1000,)
First 5 voltages (complex p.u.):
[1.        +0.00000000e+00j 0.99999708-4.54545455e-06j
 0.99999416-9.09090909e-06j 0.99999123-1.36363636e-05j
 0.99998831-1.81818182e-05j]
Magnitudes:
[1.         0.99999708 0.99999416 0.99999123 0.99998831]
